# Dự đoán điểm thi bằng hồi quy tuyến tính

Kéo các thanh trượt rồi bấm **Dự đoán**.

> **Lưu ý:** Ứng dụng dùng dữ liệu tổng hợp, chỉ phục vụ giảng dạy.
> Không dùng để đánh giá sinh viên thật.


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

MODEL_PATH = Path("artifacts/linear_regression.joblib")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
feature_names = artifact["feature_names"]


In [ ]:
style = {"description_width": "110px"}
layout = widgets.Layout(width="430px")

hours = widgets.FloatSlider(
    value=6, min=0, max=20, step=0.5, description="Giờ học:",
    continuous_update=False, style=style, layout=layout)
attendance = widgets.FloatSlider(
    value=80, min=50, max=100, step=1, description="Chuyên cần (%):",
    continuous_update=False, style=style, layout=layout)
assignments = widgets.IntSlider(
    value=7, min=0, max=10, step=1, description="Bài tập đã làm:",
    continuous_update=False, style=style, layout=layout)
previous = widgets.FloatSlider(
    value=6.5, min=0, max=10, step=0.1, description="Điểm trước:",
    continuous_update=False, style=style, layout=layout)

predict_button = widgets.Button(description="Dự đoán", button_style="primary")
reset_button = widgets.Button(description="Đặt lại")
output = widgets.Output()

DEFAULTS = [(hours, 6), (attendance, 80), (assignments, 7), (previous, 6.5)]


def input_frame():
    values = {
        "hours_studied": hours.value,
        "attendance": attendance.value,
        "assignments_completed": assignments.value,
        "previous_score": previous.value,
    }
    return pd.DataFrame(
        [[values[name] for name in feature_names]], columns=feature_names
    )


def on_predict(_):
    predict_button.disabled = True
    with output:
        output.clear_output()
        try:
            raw = float(model.predict(input_frame())[0])
            print(f"Điểm thi dự đoán: {np.clip(raw, 0, 10):.2f}/10")
        except Exception:
            print("Không thể dự đoán. Hãy kiểm tra dữ liệu và artifact.")
        finally:
            predict_button.disabled = False


def on_reset(_):
    for widget, value in DEFAULTS:
        widget.value = value
    output.clear_output()


predict_button.on_click(on_predict)
reset_button.on_click(on_reset)

display(widgets.VBox([
    hours, attendance, assignments, previous,
    widgets.HBox([predict_button, reset_button]),
    output,
]))
